# Tutorial: enumerating metabolites with Metabolic Forest

This notebook shows how to use `xenosite.forest` to generate metabolite structures and search pathways between a reactant and a product.

Site-of-metabolism *scores* are available at [xenosite.org](https://xenosite.org). This package enumerates *structures*.

Please cite Hughes et al., *Metabolic Forest*, *J. Chem. Inf. Model.* 2020, DOI [10.1021/acs.jcim.0c00360](https://doi.org/10.1021/acs.jcim.0c00360) if you use this software. Copy-paste BibTeX is in the [README](../README.md#citation). Rulesets and the papers they match (Rainbow Phase I, quinone, bioactivation) are in [docs/rulesets.md](../docs/rulesets.md).

In [1]:
from rdkit import Chem
from xenosite.forest import bfs, rules, RuleSet, PhaseOneRS, load_ruleset

## Enumerate metabolites of one rule

Hydroxylate propane and print the product SMILES.

In [2]:
mol = Chem.MolFromSmiles("CCC")
for site, products in rules.Hydroxylation().metabolites(mol):
    print(site, [Chem.MolToSmiles(p) for p in products])

('Hydroxylation_SmartsReactionRuleRxn0', frozenset({0})) ['CCCO']
('Hydroxylation_SmartsReactionRuleRxn0', frozenset({1})) ['CC(C)O']
('Hydroxylation_SmartsReactionRuleRxn0', frozenset({2})) ['CCCO']
('Hydroxylation_SmartsReactionRuleRxn1', frozenset({1})) ['CC(C)=O']


## Combine rules

`RuleSet` groups rules. `PhaseOneRS` is the Phase I collection used in Metabolic Forest.

In [3]:
print(sorted({rule.name for rule in PhaseOneRS}))

combo = RuleSet([rules.Epoxidation(), rules.EpoxideOpening()], name="epoxide")
reactant = Chem.MolFromSmiles("c1ccccc1")
product = Chem.MolFromSmiles("C1=CC=CC(O)C1O")
smiles, steps, mols = next(combo.find_path(reactant, product, depth=2))
print(smiles)
print(steps)

['Dealkylation', 'Dehydration', 'Dehydrogenation', 'Dephosphorylation', 'Epoxidation', 'EpoxideOpening', 'Hydrogenation', 'Hydrolysis', 'Hydroxylation', 'NitrogenOxidation', 'NitrogenReduction', 'OxidativeDehalogenation', 'OxygenReduction', 'ReductiveDehalogenation', 'SulfurOxidation', 'SulfurReduction']
OC1C=CC=CC1O
[('Epoxidation', frozenset({0, 1})), ('EpoxideOpening', frozenset({0, 2}))]


## Search a pathway with `bfs`

Pass reactant and product SMILES. Use `phase1=True` for Phase I site strings.

In [4]:
smiles, steps, mols = next(bfs(["CCO", "CC=O"], ruleset="PhaseOneRS", phase1=True))
print(smiles)
print(steps)

CC=O
[('Dehydrogenation', frozenset({'2.h', '3.h'}))]


## Build a metabolite network

`MetaboliteNetwork` (optional extra: `uv add "xenosite-forest[network]"`) is a directed graph of molecules. Call `expand` with a ruleset to add one generation of metabolites from every unexpanded node. Edges store the reactions that connect two structures.

In [5]:
from xenosite.forest.net import MetaboliteNetwork

net = MetaboliteNetwork("CCO")
net.expand(PhaseOneRS)
print(f"{net.number_of_nodes()} metabolites, {net.number_of_edges()} reactions")
sorted(net.nodes)

10 metabolites, 9 reactions


['C=C', 'C=CO', 'C=O', 'CC', 'CC(=O)O', 'CC(O)O', 'CC=O', 'CCO', 'CO', 'OCCO']

### Draw the network

In a notebook, evaluating `net` (or calling `net.draw()`) shows a layered figure: structures as nodes, generations as rows, and edges colored by Metabolic Rainbow class (SO red, UO orange, DH green, HD blue, RD purple).

In [6]:
net.draw()

<Drawing 23688 chars>

### Structure grid

`net.grid()` shows the same molecules as a compact structure grid, ordered by generation. Useful when the graph is dense.

In [7]:
net.grid()

<Drawing 21471 chars>

### Find and display a pathway

`paths` walks the graph from a reactant to a product. `draw_path` renders that route as a linear reaction scheme; `draw(highlight=...)` keeps the full network and emphasizes the path.

In [8]:
path = next(net.paths("CCO", "CC=O"))
print(path)
net.draw_path(path)

['CCO', 'CC=O']


<Drawing 4821 chars>

In [9]:
net.draw(highlight=path)

<Drawing 22846 chars>

### Several steps deep

Each `expand` adds one generation. Use a **small ruleset** so the graph stays drawable — full Phase I explodes quickly after a couple of generations.
Here dehydrogenation and epoxidation on propene give a clean three-generation network.

In [10]:
tiny = RuleSet([rules.Dehydrogenation(), rules.Epoxidation()], name="tiny")
deep = MetaboliteNetwork("C=CC")
deep.expand(tiny)
deep.expand(tiny)  # second generation
print(f"{deep.number_of_nodes()} metabolites, {deep.number_of_edges()} reactions")
deep.draw()

5 metabolites, 5 reactions


<Drawing 12935 chars>

In [11]:
path = next(deep.paths("C=CC", "C=C1CO1", extra_depth=2))
print(path)
deep.draw_path(path)

['C=CC', 'C=C=C', 'C=C1CO1']


<Drawing 6906 chars>

In [12]:
deep.draw(highlight=path)

<Drawing 12710 chars>

### Drawing options

Defaults are publication-friendly: **white figure background** and white molecule panels.

`draw()` options include:

- `nodes=[...]` — draw only these molecules (and edges among them)
- `reaction_types=[...]` — keep only those edge labels
- `max_generation=N` — only generations `0..N`
- `background` — figure background color
- `mol_background`, `mol_border`, `mol_border_width` — molecule panel style
- `highlight_border`, `highlight_border_width` — panel style for a highlighted path
- `mol_size`, `target_width`, `title`, `show_labels`, `show_legend`, `show_smiles`
- `highlight=path`, `fade_others=False`

The same filters work on `grid()` and `prune()`.

In [13]:
net.draw(
    reaction_types=["Dehydrogenation", "Hydroxylation"],
    title="Ethanol — oxygenation & dehydrogenation only",
    mol_size=100,
)

<Drawing 17377 chars>

In [14]:
# Limit the figure to an explicit set of molecules
net.draw(
    nodes=["CCO", "CC=O", "C=CO", "OCCO"],
    title="Selected metabolites only",
)

<Drawing 11076 chars>

In [15]:
# Style the figure and molecule panels (print-friendly defaults are already white)
net.draw(
    nodes=["CCO", "CC=O", "C=CO"],
    highlight=["CCO", "CC=O"],
    background="#ffffff",
    mol_background="#ffffff",
    mol_border="#888888",
    mol_border_width=1.5,
    highlight_border="#222222",
    highlight_border_width=2.5,
    title="Custom panel borders",
)

<Drawing 7382 chars>

In [16]:
net.draw(
    max_generation=0,
    show_legend=False,
    show_labels=False,
    title="Starting molecule only",
)

<Drawing 2284 chars>

### Prune the network

`prune` returns a filtered **copy** (pass `inplace=True` to modify the original).
You can keep a path, cap generation depth, or keep selected reaction types — then draw the smaller graph.

In [17]:
dh_net = net.prune(reaction_types=["Dehydrogenation"])
print(f"{dh_net.number_of_nodes()} metabolites, {dh_net.number_of_edges()} reactions")
dh_net.draw(title="Pruned to Dehydrogenation edges")

3 metabolites, 2 reactions


<Drawing 7404 chars>

In [18]:
path_net = net.prune(path=["CCO", "CC=O"])
path_net.draw_path(["CCO", "CC=O"])

<Drawing 4821 chars>

### Save SVG, PDF, or PostScript

`Drawing.save(path)` (or `net.save_draw(path, ...)`) writes vector figures:

- **`.svg`** — always available (no extra tools)
- **`.pdf` / `.ps` / `.eps`** — need [`rsvg-convert`](https://gitlab.gnome.org/GNOME/librsvg) (librsvg) or ImageMagick `magick`

On macOS with Homebrew: `brew install librsvg` or `brew install imagemagick`.

In [19]:
from pathlib import Path
import tempfile

tmpdir = Path(tempfile.mkdtemp(prefix="forest-figures-"))
fig = net.draw(reaction_types=["Dehydrogenation"], title="Dehydrogenation")

fig.save(tmpdir / "ethanol_dh.svg")  # always works
print("SVG:", tmpdir / "ethanol_dh.svg")

for ext in ("pdf", "ps"):
    dest = tmpdir / f"ethanol_dh.{ext}"
    try:
        fig.save(dest)
        print(f"{ext.upper()}:", dest, f"({dest.stat().st_size} bytes)")
    except RuntimeError as err:
        print(f"{ext.upper()} unavailable:", err)

# Equivalent one-liner:
# net.save_draw(tmpdir / "ethanol_dh.svg", reaction_types=["Dehydrogenation"])

SVG: /var/folders/js/kzk69ksj06x2v95lx3r4ckc00000gn/T/forest-figures-tt3wmmk9/ethanol_dh.svg
PDF: /var/folders/js/kzk69ksj06x2v95lx3r4ckc00000gn/T/forest-figures-tt3wmmk9/ethanol_dh.pdf (32832 bytes)
PS: /var/folders/js/kzk69ksj06x2v95lx3r4ckc00000gn/T/forest-figures-tt3wmmk9/ethanol_dh.ps (88006 bytes)
